In [ ]:
import uuid
from datetime import datetime
from agent.basemodels import InitialInput,Attachment, Event,FactSheet
from agent import ContextManager
from langchain_google_genai import ChatGoogleGenerativeAI
from pathlib import Path
from dotenv import load_dotenv
import os
import logging
logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)
load_dotenv() #, os.getenv("GOOGLE_API_KEY"), os.getcwd()

True

In [2]:
from pydantic import BaseModel

from typing import Literal, Optional
class AttachmentModel(BaseModel):
    """Attachment sent to backend API"""
    filename: str
    file_id: str
    content: str  # Base64 for PDF, text for others
    file_type: str
    size: int
class AskAgentRequest(BaseModel):
    """POST /ask-agent request"""
    question: str
    attachments: list[AttachmentModel]
    session_id: str
    domain: str
    agent_type: Literal["fast", "expert"]
    llm_provider: Literal["google", "openai", "claude"]
    query_id: str
    project_id: Optional[str] = None

In [3]:
nils_christine = """Vi (Bahr) representerer Anders og Berit Kristiansen som kjøpte eiendommen Fjellveien 42A i Stavanger fra Carl Danielsen for 7 030 000 kroner i juni 2019.
Kort tid etter overtakelse oppsto lekkasjer i tilbygget. Undersøkelser avdekket omfattende avvik: feilkonstruert betongdekke, tilbygg oppført delvis utenfor eiendomsgrensen i strid med byggetillatelse, og flere ulovlige murer - én av dem over nabogrensen.
Våre klienter krevde heving 21. juni 2021. Etter avslag tok vi ut stevning 17. november 2021 med krav om heving og erstatning, subsidiært prisavslag og erstatning."""

sven_kare = """Vi (Bahr) representerer Sven Kåre Sture som solgte eiendommen Fjellveien 42A i Stavanger til Anders og Berit Kristiansen for 7 030 000 kroner i juni 2019. Eiendommen ble solgt "som den er".
Kjøperne hevder det foreligger omfattende avvik ved tilbygget, betongdekket og enkelte murer på eiendommen. De krevde heving 21. juni 2021.
Vår klient bestrider at avvikene gir grunnlag for heving. Han hadde ikke kunnskap om de påståtte forholdene ved salget, og eiendommen er ikke i vesentlig dårligere stand enn kjøperne hadde grunn til å forvente. Partene inngikk dessuten forliksavtale i mars 2022 hvor vår klient påtok seg søknadsprosessen og prisavslag.
Vi anfører at prisavslag er tilstrekkelig virkemiddel, og at heving er avskåret."""

attachments = []
for idx, file in enumerate(Path("/Users/sigvardbratlie/Documents/Projects/master-thesis/data/THRD-2021-163881/fabricated").glob("*")):
    with open(file, "r") as f:
        content = f.read()
    
    attachments.append(AttachmentModel(
        filename=file.name,
        file_id=str(uuid.uuid4()),
        content=content,
        file_type="text/plain", 
        size=os.path.getsize(file)
    ))

    if idx >= 1:
        break

request = AskAgentRequest(
    question = nils_christine,
    attachments = attachments,
    session_id = str(uuid.uuid4()),
    domain = "legal",
    agent_type = "fast",
    llm_provider = "google",
    query_id = str(uuid.uuid4()),
    project_id = str(uuid.uuid4()))

In [ ]:
context_manager = ContextManager(llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0))
vs = "VectorSearch()"

In [ ]:
user_input = request.question
attachments = [att.model_dump() for att in request.attachments]

events = []
damages = []
claims = []
deadlines = []
files = []

initial_input = await context_manager.analyze_init_input(user_input)
for att in attachments:
    logger.debug(f"Analyzing attachment: {att.get('filename','')} (ID: {att.get('file_id','')})")
    content_txt = att.get("content", "")
    result = await context_manager.analyze_doc(initial_input, content_txt, 
                                                    file_id=att.get("file_id",""),
                                                    filename=att.get("filename",""),
                                                    path="",
                                                    file_type=att.get("file_type",""),
                                                    size=len(content_txt),
                                                    )
    analyzed_doc = result.get("file")
    logger.debug(f"Analyzed document: {analyzed_doc.filename} (ID: {analyzed_doc.file_id}) - Result {analyzed_doc.model_dump()}")

    # Collect results from analyzed documents
    files.append(analyzed_doc)
    damages.extend(analyzed_doc.damage) if analyzed_doc.damage else None
    claims.extend(analyzed_doc.claim) if analyzed_doc.claim else None
    deadlines.extend(analyzed_doc.deadline) if analyzed_doc.deadline else None

    events.extend(result.get("events", [])) if result.get("events") else None

factual_facts = await context_manager.analyze_factual_facts(initial_input, events)

events_txt = " ".join([f"- {event.description} (Date: {event.date}" for event in events])
rag_content_law = vs.query(query = events_txt, table_name = "laws", n_results=2) #Implement query based on initial input
governing_law = await context_manager.analyze_governing_law(events = events, rag_content_law=rag_content_law) #IMplementer

result = FactSheet(timeline=events,
                    damages=damages,
                    claims=claims,
                    deadlines=deadlines,
                    governing_law=governing_law,
                    **factual_facts.model_dump(),
                    **initial_input.model_dump(),
                    )

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents "HTTP/1.1 200 OK"
INFO:langchain_google_community.bq_storage_vectorstores._base:BigQuery table master-thesis-26.vector_store.laws initialized/validated as persistent storage. Access via BigQuery console:
 https://console.cloud.google.com/bigquery?project=master-thesis-26&ws=!1m5!1m4!4m3!1smaster-thesis-26!2svector_store!3slaws
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents "HTTP/1.1 200 OK"
INF

In [7]:
result.model_dump()

{'disputed_facts': ['The existence and extent of extensive defects in the property Fjellveien 42A, including an incorrectly constructed concrete slab, an extension built partially outside the property boundary in violation of building permits, and several illegal walls.',
  'The exact date or period when Anders and Berit Kristiansen discovered the defects in the property Fjellveien 42A.',
  'Whether the declaration of rescission on June 21, 2021, was made within a reasonable time after the defects were discovered.'],
 'undisputed_facts': ['Anders and Berit Kristiansen purchased the property Fjellveien 42A in Stavanger from Carl Danielsen for 7,030,000 NOK in June 2019.',
  'Leaks occurred in the extension shortly after Anders and Berit Kristiansen took possession.',
  'Anders and Berit Kristiansen formally declared the rescission of the property purchase on June 21, 2021.',
  'The rescission declaration demanded repayment of the purchase price and damages totaling 9,778,904 NOK.',
  'F

In [ ]:
conversation_manager = ""

DEBUG:google.auth._default:Checking gcloud-keys.json for explicit credentials as part of auth process...


In [5]:
db = conversation_manager.db
user_id = "p5hsFGQaM2adW8sPG9T3"
project_id = "5f37dbb0-75a9-4382-ac82-2d14a534d19b"
try:
    factsheet_ref = (
        db.collection("projects")
            .document(user_id)
            .collection("factsheets")
            .document(project_id)
    )
    
    factsheet_doc = factsheet_ref.get()
    
    if not factsheet_doc.exists:
        logger.warning(f"No factsheet found for project_id: {project_id}")
    
    factsheet_data = factsheet_doc.to_dict()
    

except Exception as e:
    logger.error(f"Error loading factsheet: {e}")


In [6]:
factsheet_data

{'agent_type': 'fast',
 'llm_provider': 'openai',
 'factsheet': {'parties': [{'entity_type': 'individual',
    'legal_representation': 'Bahr',
    'key_contact': None,
    'party_id': 'P1',
    'legal_name': 'Anders Kristiansen',
    'role': 'plaintiff'},
   {'entity_type': 'individual',
    'legal_representation': 'Bahr',
    'key_contact': None,
    'party_id': 'P2',
    'legal_name': 'Berit Kristiansen',
    'role': 'plaintiff'},
   {'entity_type': 'individual',
    'legal_representation': None,
    'key_contact': None,
    'party_id': 'P3',
    'legal_name': 'Carl Danielsen',
    'role': 'defendant'}],
  'claims': [{'relief_sought': 'Rescission of the purchase agreement and compensation, or subsidiarily, price reduction and compensation.',
    'factual_basis': 'Anders and Berit Kristiansen purchased Fjellveien 42A from Carl Danielsen on June 1, 2019, for 7,030,000 NOK. Subsequent to taking possession, extensive defects were discovered, including a wrongly constructed concrete slab,

In [4]:
from pydantic import BaseModel
class Test(BaseModel):
    test1: str
    test2: int

t = Test(test1="example", test2=42)
t2 = {"test1" : 10, "test2" : t}

In [2]:
from langchain_core.runnables import RunnableConfig
c = RunnableConfig(configurable={"user_id": "1234", "custom_project_id": "5678"})

In [ ]:
from google.cloud import firestore
db = firestore.Client()
user_id = "p5hsFGQaM2adW8sPG9T3"
projects_ref = (
    db.collection("projects")
    .document(user_id)
    .collection("projects")
)

projects_docs = projects_ref.stream()

all_projects = []

for project_doc in projects_docs:
    project_data = project_doc.to_dict()
    project_id = project_doc.id
    
    all_projects.append({
        "project_id": project_id,
        "title": project_data.get("factsheet",{}).get("title", ""),
        "created_at": project_data.get("created_at"),
    })

# Sorter etter created_at (nyeste først)
all_projects.sort(
    key=lambda x: x.get("created_at") or "", 
    reverse=True
)


In [3]:
from agent.basemodels import Deadline

In [23]:
d = Deadline(
    description="Submit final report",
    date="2024-12-31",
    file_id="file_12345",
    responsible_party="John Doe"
)
a = Deadline(
    description="Prepare presentation",
    date="2024-11-30",
    file_id="file_67890",
    responsible_party="Jane Smith"
)
l = [d, a]

In [ ]:
for item in l:
    a = item.model_dump("json")
    d = a.pop("date")

[{'date': '2024-12-31T00:00:00',
  'description': 'Submit final report',
  'file_id': 'file_12345',
  'responsible_party': 'John Doe'},
 {'date': '2024-11-30T00:00:00',
  'description': 'Prepare presentation',
  'file_id': 'file_67890',
  'responsible_party': 'Jane Smith'}]

In [21]:
a = a.model_dump(exclude = "date")

In [24]:
Deadline.model_validate(a)

Deadline(date=datetime.datetime(2024, 11, 30, 0, 0), description='Prepare presentation', file_id='file_67890', responsible_party='Jane Smith')